In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import os
from src.audit import (
    compute_checksum,
    validate_schema,
    audit_dataframe,
    compare_audits,
    validate_data_quality
)

print("=" * 60)
print("PRUEBA COMPLETA DE AUDIT.PY")
print("=" * 60)

# ============================================
# 1. Crear datos de prueba
# ============================================
print("\n1. CREANDO DATOS DE PRUEBA")

df_original = pd.DataFrame({
    'age': [25, 30, 35, 1000, 28, 32],
    'job': ['engineer', 'unknown', 'doctor', 'engineer', '', 'teacher'],
    'salary': [50000, 60000, 55000, np.nan, 52000, 58000],
    'constant': [5, 5, 5, 5, 5, 5]
})

print("DataFrame creado:")
print(df_original)

# ============================================
# 2. Probar compute_checksum
# ============================================
print("\n" + "=" * 60)
print("2. PRUEBA DE COMPUTE_CHECKSUM")
print("=" * 60)

checksum1 = compute_checksum(df_original)
checksum2 = compute_checksum(df_original)
print(f"Checksum 1: {checksum1}")
print(f"Checksum 2: {checksum2}")
print(f"✅ ¿Son iguales? {checksum1 == checksum2}")

# ============================================
# 3. Probar validate_schema
# ============================================
print("\n" + "=" * 60)
print("3. PRUEBA DE VALIDATE_SCHEMA")
print("=" * 60)

expected = ['age', 'job', 'salary', 'constant']
result = validate_schema(df_original, expected)
print(f"Esquema esperado: {expected}")
print(f"✅ ¿Válido? {result['valid']}")

# ============================================
# 4. Probar audit_dataframe
# ============================================
print("\n" + "=" * 60)
print("4. PRUEBA DE AUDIT_DATAFRAME")
print("=" * 60)

# Limpiar archivo de log anterior
if os.path.exists('outputs/audit_log.json'):
    os.remove('outputs/audit_log.json')

audit_result = audit_dataframe(
    df_original, 
    'raw_input',
    expected_columns=['age', 'job', 'salary']
)

print("\n✅ Auditoría completada")

# ============================================
# 5. Simular limpieza y comparar
# ============================================
print("\n" + "=" * 60)
print("5. SIMULAR LIMPIEZA")
print("=" * 60)

df_clean = df_original.copy()
df_clean = df_clean.drop(columns=['constant'])
df_clean['job'] = df_clean['job'].replace(['unknown', ''], np.nan)
df_clean['salary'] = df_clean['salary'].fillna(df_clean['salary'].median())
df_clean['age'] = df_clean['age'].clip(upper=100)

print("DataFrame limpio:")
print(df_clean)

audit_clean = audit_dataframe(df_clean, 'after_cleaning')

# ============================================
# 6. Comparar auditorías
# ============================================
print("\n" + "=" * 60)
print("6. COMPARAR AUDITORÍAS")
print("=" * 60)

comparison = compare_audits(audit_result, audit_clean)

# ============================================
# 7. Validar calidad de datos
# ============================================
print("\n" + "=" * 60)
print("7. VALIDAR CALIDAD DE DATOS")
print("=" * 60)

numeric_bounds = {'age': (0, 120), 'salary': (30000, 200000)}
categorical_values = {'job': ['engineer', 'doctor', 'teacher']}

quality_result = validate_data_quality(
    df_clean, 
    numeric_bounds=numeric_bounds,
    categorical_values=categorical_values
)

# ============================================
# 8. Verificar archivo de log
# ============================================
print("\n" + "=" * 60)
print("8. VERIFICAR ARCHIVO DE LOG")
print("=" * 60)

if os.path.exists('outputs/audit_log.json'):
    with open('outputs/audit_log.json', 'r') as f:
        lines = f.readlines()
    print(f"✅ Archivo de log creado con {len(lines)} entradas")
    for i, line in enumerate(lines, 1):
        print(f"   Entrada {i}: {line[:80]}...")
else:
    print("❌ Archivo de log no encontrado")

# ============================================
# Resumen final
# ============================================
print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print("✅ compute_checksum: OK")
print("✅ validate_schema: OK")
print("✅ audit_dataframe: OK")
print("✅ compare_audits: OK")
print("✅ validate_data_quality: OK")
print("\n🎉 Módulo audit.py funcionando correctamente!")

PRUEBA COMPLETA DE AUDIT.PY

1. CREANDO DATOS DE PRUEBA
DataFrame creado:
    age       job   salary  constant
0    25  engineer  50000.0         5
1    30   unknown  60000.0         5
2    35    doctor  55000.0         5
3  1000  engineer      NaN         5
4    28            52000.0         5
5    32   teacher  58000.0         5

2. PRUEBA DE COMPUTE_CHECKSUM
Checksum 1: 4deaec149fdf6481df77d44a60866f03
Checksum 2: 4deaec149fdf6481df77d44a60866f03
✅ ¿Son iguales? True

3. PRUEBA DE VALIDATE_SCHEMA
Esquema esperado: ['age', 'job', 'salary', 'constant']
✅ ¿Válido? True

4. PRUEBA DE AUDIT_DATAFRAME
[Audit] Advertencia: No se pudo guardar el log: Object of type int64 is not JSON serializable

AUDITORÍA: RAW_INPUT
  Shape: 6 filas × 4 columnas
  Nulos totales: 1
  Checksum: 4deaec149fdf6481...
  Esquema válido: ✅ Sí
  Columnas extra detectadas (no hay problema): ['constant']...

  ⚠️ Advertencias (2):
     - Columnas numéricas constantes: ['constant']
     - Valores 'unknown' detectados: